# 02. 고정 비율과 발견한 breakpoint 비교

논문의 LayerNorm 고정 비율 60%와 01에서 찾은 데이터셋별 breakpoint를 비교한다. 두 설정 모두 총 4,800 step을 사용하고, 비율에 따라 Stage 1과 Stage 2의 step만 나눈다.

## 실험 설계

- **Fixed**: Stage 1을 2,880 step(60%), Stage 2를 1,920 step 학습한다.
- **Found**: 01에서 선택한 step까지 Stage 1을 학습하고 나머지를 Stage 2에 사용한다.
- 두 run은 같은 seed의 fresh model에서 시작한다.
- Test label은 학습이나 비율 선택에 사용하지 않고 최종 평가에서만 읽는다.
- `B`, `N`, `H`는 Base 정확도, Novel 정확도, 두 값의 조화평균이다.

## 1. Imports

노트북에는 실험의 핵심 흐름을 두고, manifest·학습 loop·artifact 입출력처럼 반복되는 기능만 `utils`에서 가져온다.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path[:0] = [str(ROOT), str(ROOT / "notebook")]

from utils import data, experiment, results as result_utils

## 2. 실험 설정

01과 같은 shot·PEFT 설정을 사용한다. Batch size, learning rate, seed 등 나머지 설정은 breakpoint JSON에서 복원해 두 실험을 동일 조건으로 맞춘다. `RESTART = True`이면 저장된 Fixed/Found 결과를 무시하고 다시 학습한다.

In [ ]:
DATASETS = data.DATASETS
SHOTS, PEFT = 16, "ln"
RESTART = False

## 3. Breakpoint 입력 확인

01에서 저장한 세 데이터셋의 선택 step과 비율을 먼저 확인한다.

In [ ]:
print(result_utils.breakpoint_summary(PEFT, SHOTS))

dataset       | val  | auto step | auto ratio | best H | selected | source   
--------------+------+-----------+------------+--------+----------+----------
eurosat       | 5400 | 80        | 0.0167     | 0.8229 | 0.0167   | automatic
fgvc_aircraft | 3333 | 1430      | 0.2979     | 0.4035 | 0.2979   | automatic
dtd           | 1128 | 3340      | 0.6958     | 0.7060 | 0.6958   | automatic


## 4. 핵심 함수: 2-stage 학습과 최종 평가

`run_two_stage`가 실제 비교 실험 한 번을 수행한다. Stage 1은 CLIP을 PEFT로 적응시키고, Stage 2는 초기화한 Base classifier만 학습한다. 모든 학습이 끝난 다음에만 test manifest와 정답을 열어 B/N/H와 submission 예측을 만든다.

In [ ]:
def run_two_stage(config, dataset, ratio):
    """Train a fresh two-stage model, then evaluate it once."""
    experiment.seed_everything(config.seed)
    device = experiment.resolve_device(config)
    train_transform, test_transform = data.clip_transforms()
    train_data = data.load_train_dataset(config, dataset, train_transform)
    method, parameters = experiment.prepare_method(
        config, train_data, device
    )

    stage_one_steps = int(config.total_steps * ratio)
    stage_two_steps = config.total_steps - stage_one_steps

    method.train()
    experiment.train_steps(
        method.stage_one_logits,
        parameters,
        experiment.make_loader(
            train_data, config, shuffle=True, seed_offset=101
        ),
        stage_one_steps,
        config,
        device,
        f"{dataset} stage1 ratio={ratio:.4f}",
    )

    method.initialize_classifier()
    method.eval()
    experiment.seed_everything(config.seed + 1)
    experiment.train_steps(
        method.stage_two_logits,
        [method.classifier],
        experiment.make_loader(
            train_data, config, shuffle=True, seed_offset=202
        ),
        stage_two_steps,
        config,
        device,
        f"{dataset} stage2 ratio={ratio:.4f}",
    )

    test_base, test_novel, ids = data.load_test_datasets_for_evaluation(
        config, dataset, test_transform
    )
    with torch.inference_mode():
        novel_classifier = method.encode_classnames(test_novel.classes)
    metrics = experiment.evaluate(
        method,
        (test_base, test_novel),
        (method.classifier, novel_classifier),
        config,
        device,
        collect_predictions=True,
    )
    result = experiment.build_comparison_result(
        config, dataset, ratio, stage_one_steps, stage_two_steps, metrics, ids
    )
    print(
        f"{dataset} ratio={ratio:.4f}: "
        f"B={metrics['base_accuracy']:.4f} "
        f"N={metrics['novel_accuracy']:.4f} "
        f"H={metrics['harmonic_mean']:.4f}"
    )

    del parameters, method
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result

## 5. Fixed/Found 실행과 checkpoint

각 데이터셋의 breakpoint 설정을 그대로 복원한 뒤 Fixed와 Found를 차례로 실행한다. 완전한 예측이 이미 저장돼 있으면 해당 run만 재사용한다.

In [ ]:
def compare_dataset(dataset, restart=False):
    breakpoint_path = experiment.artifact_path(
        dataset, PEFT, SHOTS, "breakpoints"
    )
    breakpoints = experiment.load_json(breakpoint_path)
    config = experiment.config_from_breakpoints(breakpoints, dataset)
    if breakpoints["signature"] != experiment.breakpoint_signature(config):
        raise ValueError("Breakpoint and comparison settings do not match")

    found_ratio = float(breakpoints["datasets"][dataset]["selected_ratio"])
    path, payload = experiment.open_comparison_checkpoint(
        config, {dataset: found_ratio}, restart
    )
    runs = payload["runs"].setdefault(dataset, {})

    for label, ratio in (("fixed", config.fixed_ratio), ("found", found_ratio)):
        if label in runs and experiment.has_complete_predictions(
            runs[label], dataset
        ):
            print(
                f"{dataset}/{label}: reuse {path.relative_to(ROOT)}"
            )
            continue
        runs[label] = run_two_stage(config, dataset, ratio)
        payload["summary"] = experiment.comparison_summary(payload)
        experiment.save_json(path, payload)

    row = payload["summary"]["datasets"][dataset]
    print(f"{dataset}: delta H={row['delta_harmonic_mean']:+.4f}")
    return payload

## 6. 세 데이터셋 실행

노트북에 정의한 함수를 현재 kernel에서 직접 호출한다. CUDA와 seed를 안전하게 분리하기 위해 한 GPU에서는 순차 실행한다.

In [ ]:
comparison_runs = {
    dataset: compare_dataset(dataset, RESTART) for dataset in DATASETS
}

eurosat/fixed: reuse results/logs/kaggle_breakpoint/comparison_eurosat_ln_16shot.json
eurosat/found: reuse results/logs/kaggle_breakpoint/comparison_eurosat_ln_16shot.json
eurosat: delta H=+0.0503
fgvc_aircraft/fixed: reuse results/logs/kaggle_breakpoint/comparison_fgvc_aircraft_ln_16shot.json
fgvc_aircraft/found: reuse results/logs/kaggle_breakpoint/comparison_fgvc_aircraft_ln_16shot.json
fgvc_aircraft: delta H=-0.0006
dtd/fixed: reuse results/logs/kaggle_breakpoint/comparison_dtd_ln_16shot.json
dtd/found: reuse results/logs/kaggle_breakpoint/comparison_dtd_ln_16shot.json
dtd: delta H=+0.0010


## 7. 최종 성능표

`delta B`, `delta N`, `delta H`는 모두 `Found - Fixed`다. 아래 표는 저장된 comparison JSON에서 다시 읽은 실제 결과다.

In [ ]:
print(result_utils.comparison_summary(PEFT, SHOTS))

dataset       | fixed r | found r | fixed B | found B | delta B | fixed N | found N | delta N | fixed H | found H | delta H
--------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------
eurosat       | 0.6000  | 0.0167  | 0.9540  | 0.9526  | -0.0014 | 0.6441  | 0.7187  | 0.0746  | 0.7690  | 0.8193  | 0.0503 
fgvc_aircraft | 0.6000  | 0.2979  | 0.4856  | 0.4748  | -0.0108 | 0.3815  | 0.3875  | 0.0060  | 0.4273  | 0.4267  | -0.0006
dtd           | 0.6000  | 0.6958  | 0.8399  | 0.8445  | 0.0046  | 0.6425  | 0.6413  | -0.0012 | 0.7281  | 0.7290  | 0.0010 

fixed mean-H=0.6415, found mean-H=0.6584, delta mean-H=+0.0169


## 8. 결과 해석

- EuroSAT은 Novel 정확도가 7.46 pp 올라 H가 5.03 pp 개선됐다.
- FGVC-Aircraft의 H는 0.06 pp 하락하고 DTD는 0.10 pp 상승해 사실상 비슷했다.
- 세 데이터셋의 macro H는 64.15에서 65.84로 1.69 pp 개선됐지만, 개선분은 EuroSAT이 주도했다.
- 결과는 seed 2026 단일 실행이므로 일반화 결론에는 다중 seed 반복이 필요하다.

## 9. Kaggle submission 생성

저장된 예측을 `test.csv` 순서로 검증·병합해 Fixed와 Found submission을 만든다.

In [ ]:
submissions = result_utils.export_model_submissions(PEFT, SHOTS)
for label, path in submissions.items():
    print(f"{label}: {path.relative_to(ROOT)}")

fixed: results/submission_fixed_ratio_0.60.csv
found: results/submission_found_breakpoints.csv


## 10. Breakpoint 곡선

색상별 굵은 세로선이 각 데이터셋에서 선택한 breakpoint다. y축을 확대한 개별 그림과 성능표는 [`results/report/README.md`](../results/report/README.md)에서 확인한다.

![세 데이터셋 breakpoint 곡선](../results/report/breakpoints_combined.png)